# PolyWhisper — Common Voice Training
Real training data. Run all cells. Re-run to resume from last checkpoint.

In [ ]:
# CELL 1: Install
!pip install -q transformers==4.44.2 datasets==3.1.0 soundfile librosa

import torch, torch.nn as nn, numpy as np, json, os, time, math
from pathlib import Path
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
from google.colab import drive

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# CELL 2: Config + Model

WHISPER_MODEL = "openai/whisper-base"
LANGUAGES = ["en", "hi"]
NUM_EPOCHS = 5
BATCH_SIZE = 8
LR = 5e-4
MAX_AUDIO_SEC = 30.0
MAX_LABEL_LEN = 256
AD_HID = 256
AD_LAYERS = 3
AD_HEADS = 4
AD_FFN = 1024
AD_RANK = 16
MAX_SAMPLES_PER_LANG = 50000

drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive/polywhisper")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = SAVE_DIR / "adapters"
ADAPTER_DIR.mkdir(exist_ok=True)
DATA_DIR = SAVE_DIR / "data_cv"
DATA_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = SAVE_DIR / "cv_training_state.json"

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
ENCODE_DIM = 512
VOCAB_SIZE = len(processor.tokenizer)
tok = processor.tokenizer

class LowRank(nn.Module):
    def __init__(self, i, o, r):
        super().__init__()
        self.a = nn.Linear(i, r, bias=False)
        self.b = nn.Linear(r, o, bias=False)
        nn.init.zeros_(self.b.weight)
    def forward(self, x): return self.b(self.a(x))

class SelfAttn(nn.Module):
    def __init__(self, d, h, r):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.o = nn.Linear(d, d)
        self.qa = LowRank(d, d, r)
        self.va = LowRank(d, d, r)
        self.h, self.dh = h, d // h
        self.sc = math.sqrt(self.dh)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x) + self.qa(x)
        k, v = self.k(x), self.v(x) + self.va(x)
        def rs(t): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = rs(q), rs(k), rs(v)
        a = self.drop(torch.softmax((q @ k.transpose(-2, -1)) / self.sc, dim=-1))
        return self.o((a @ v).transpose(1, 2).contiguous().view(B, T, -1))

class CrossAttn(nn.Module):
    def __init__(self, d, ed, h, r):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(ed, d)
        self.v = nn.Linear(ed, d)
        self.o = nn.Linear(d, d)
        self.qa = LowRank(d, d, r)
        self.va = LowRank(ed, d, r)
        self.h, self.dh = h, d // h
        self.sc = math.sqrt(self.dh)
        self.drop = nn.Dropout(0.1)
    def forward(self, x, enc):
        B, Td, _ = x.shape
        Te = enc.size(1)
        q = self.q(x) + self.qa(x)
        k, v = self.k(enc), self.v(enc) + self.va(enc)
        def rs(t, T): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = rs(q, Td), rs(k, Te), rs(v, Te)
        a = self.drop(torch.softmax((q @ k.transpose(-2, -1)) / self.sc, dim=-1))
        return self.o((a @ v).transpose(1, 2).contiguous().view(B, Td, -1))

class AdaptLayer(nn.Module):
    def __init__(self, d, ed, h, ffn, r):
        super().__init__()
        self.sa = SelfAttn(d, h, r)
        self.ca = CrossAttn(d, ed, h, r)
        self.ff = nn.Sequential(nn.Linear(d, ffn), nn.GELU(), nn.Dropout(0.1), nn.Linear(ffn, d))
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.n3 = nn.LayerNorm(d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x, enc):
        x = x + self.drop(self.sa(self.n1(x)))
        x = x + self.drop(self.ca(self.n2(x), enc))
        x = x + self.drop(self.ff(self.n3(x)))
        return x

class LangAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.te = nn.Embedding(VOCAB_SIZE, AD_HID)
        self.pe = nn.Embedding(MAX_LABEL_LEN, AD_HID)
        self.layers = nn.ModuleList([AdaptLayer(AD_HID, ENCODE_DIM, AD_HEADS, AD_FFN, AD_RANK) for _ in range(AD_LAYERS)])
        self.out = nn.Linear(AD_HID, VOCAB_SIZE, bias=False)
        self.out.weight = self.te.weight
        self.norm = nn.LayerNorm(AD_HID)
        self.drop = nn.Dropout(0.1)
    def forward(self, enc, ids):
        ids = ids.clamp(min=0, max=VOCAB_SIZE - 1)
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        x = self.drop(self.te(ids) + self.pe(pos))
        for l in self.layers:
            x = l(x, enc)
        return self.out(self.norm(x))

class PolyWhisper(nn.Module):
    def __init__(self):
        super().__init__()
        self.whisper = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)
        for p in self.whisper.model.encoder.parameters():
            p.requires_grad = False
        self.adapters = nn.ModuleDict({l: LangAdapter() for l in LANGUAGES})
    def encode(self, feat):
        return self.whisper.model.encoder(feat).last_hidden_state
    def forward(self, feat, dec_ids, lang):
        enc = self.encode(feat)
        return self.adapters[lang](enc, dec_ids)
    def load_ckpt(self, path, device="cpu"):
        st = torch.load(path, map_location=device, weights_only=True)
        for n, s in st.items():
            if n in self.adapters:
                self.adapters[n].load_state_dict(s)

model = PolyWhisper().to("cuda")
opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable/1e6:.1f}M")

In [ ]:
# CELL 3: Load Common Voice

import soundfile as sf

MAX_AUDIO_SAMPLES = int(MAX_AUDIO_SEC * 16000)

class CommonVoice:
    def __init__(self, lang, split="train"):
        self.lang = lang
        self.split = split
        self.cache = DATA_DIR / f"cv_{lang}_{split}.json"
        self.adir = DATA_DIR / f"audio_{lang}_{split}"
        self.adir.mkdir(parents=True, exist_ok=True)
        self.data = self._load()
    def _load(self):
        if self.cache.exists():
            print(f"  Cached {self.lang}/{self.split}")
            return json.load(open(self.cache))
        cfg_map = {"en": "en", "hi": "hi"}
        print(f"  Downloading Common Voice {self.lang}...")
        ds = load_dataset(
            "mozilla-foundation/common_voice_17_0",
            cfg_map[self.lang],
            split=self.split,
            streaming=True,
        )
        recs = []
        for i, item in enumerate(tqdm(ds, desc=f"  {self.lang}", total=MAX_SAMPLES_PER_LANG)):
            if i >= MAX_SAMPLES_PER_LANG:
                break
            try:
                audio = item["audio"]["array"]
                sr = item["audio"]["sampling_rate"]
                if sr != 16000:
                    import librosa
                    audio = librosa.resample(np.array(audio, dtype=np.float32), orig_sr=sr, target_sr=16000)
                audio = np.array(audio, dtype=np.float32)
                if len(audio) < 1600 or len(audio) > MAX_AUDIO_SAMPLES:
                    continue
                text = item["sentence"].strip()
                if not text or len(text) < 3:
                    continue
                wav_path = self.adir / f"{i:06d}.wav"
                sf.write(str(wav_path), audio, 16000)
                recs.append({"wav": str(wav_path), "text": text.lower() if self.lang == "en" else text})
            except Exception:
                continue
        json.dump(recs, open(self.cache, "w"))
        print(f"  Saved {len(recs)} samples")
        return recs
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        r = self.data[i]
        a, _ = sf.read(r["wav"])
        return {"audio": a, "text": r["text"]}

print("Loading Common Voice datasets...")
train_sets, test_sets = {}, {}
for lang in LANGUAGES:
    train_sets[lang] = CommonVoice(lang, "train")
    test_sets[lang] = CommonVoice(lang, "test")
    print(f"  {lang}: {len(train_sets[lang])} train, {len(test_sets[lang])} test")

In [ ]:
# CELL 4: DataLoader

from torch.utils.data import Dataset, DataLoader

class Collator:
    def __init__(self, proc, max_aud, max_lbl):
        self.proc = proc
        self.max_aud = max_aud
        self.max_lbl = max_lbl
    def __call__(self, batch):
        auds, feats, lbls = [], [], []
        for item in batch:
            auds.append(item["audio"])
        ft = self.proc.feature_extractor(auds, sampling_rate=16000, return_tensors="pt", padding=True)
        feats = ft["input_features"]
        max_t = feats.size(-1)
        if max_t > self.max_aud:
            max_t = self.max_aud
        feats = feats[:, :, :max_t]
        for item in batch:
            toks = self.proc.tokenizer(
                item["text"], padding="max_length", max_length=self.max_lbl, truncation=True
            )
            lbls.append(toks["input_ids"])
        lbls = torch.tensor(lbls, dtype=torch.long)
        lbls[lbls == tok.pad_token_id] = -100
        return {"feats": feats, "lbls": lbls}

collator = Collator(processor, int(MAX_AUDIO_SEC * 16000 / 320 + 1), MAX_LABEL_LEN)

def make_loader(dataset, batch_size=16, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collator, num_workers=2, pin_memory=True)

print("Data pipeline ready")

In [ ]:
# CELL 5: Training Loop

def load_state():
    if STATE_FILE.exists():
        st = json.load(open(STATE_FILE))
        model.load_ckpt(str(ADAPTER_DIR / f"cv_{st['lang']}_ep{st['epoch']}_last.pt"))
        opt.load_state_dict(st["opt"])
        print(f"  Resumed: {st['lang']} epoch {st['epoch']}, step {st['step']}")
        return st
    return {"lang": LANGUAGES[0], "epoch": 0, "step": 0, "train_i": {l: 0 for l in LANGUAGES}, "val_i": {l: 0 for l in LANGUAGES}}

def save_state(lang, epoch, step, train_i, val_i):
    json.dump({"lang": lang, "epoch": epoch, "step": step, "opt": opt.state_dict(), "train_i": train_i, "val_i": val_i}, open(STATE_FILE, "w"))

def save_ckpt(lang, epoch, tag):
    ckpt = ADAPTER_DIR / f"cv_{lang}_ep{epoch + 1}_{tag}.pt"
    torch.save({n: a.state_dict() for n, a in model.adapters.items()}, str(ckpt))
    print(f"  Saved: {ckpt.name}")
    return ckpt

def save_best(lang, val_loss, epoch):
    p = ADAPTER_DIR / f"cv_{lang}_best.pt"
    torch.save({lang: model.adapters[lang].state_dict()}, str(p))
    print(f"  New best: {p.name}")
    return p

state = load_state()
start_lang_i = LANGUAGES.index(state["lang"])

for ep in range(state["epoch"], NUM_EPOCHS):
    for li in range(start_lang_i, len(LANGUAGES)):
        lang = LANGUAGES[li]
        train_i = state["train_i"][lang]
        loader = make_loader(train_sets[lang], batch_size=BATCH_SIZE)
        model.train()
        total, steps = 0.0, 0
        print(f"\nEpoch {ep+1}/{NUM_EPOCHS} | Training {lang.upper()} from {train_i}")

        for batch in loader:
            if steps < train_i:
                steps += 1
                continue
            feats = batch["feats"].to("cuda")
            lbls = batch["lbls"].to("cuda")
            logits = model(feats, lbls[:, :-1], lang)
            loss = nn.functional.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), lbls[:, 1:].reshape(-1), ignore_index=-100
            )
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
            steps += 1
            if steps % 50 == 0:
                avg = total / 50
                print(f"  [{steps}/{len(loader)}] loss={avg:.4f}")
                total = 0.0
                save_state(lang, ep, steps, {l: (steps if l == lang else 0) for l in LANGUAGES}, state["val_i"])

        save_ckpt(lang, ep, "last")
        del loader
        import gc; gc.collect(); torch.cuda.empty_cache()

        val_loader = make_loader(test_sets[lang], batch_size=BATCH_SIZE, shuffle=False)
        model.eval()
        val_loss, val_n = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                feats = batch["feats"].to("cuda")
                lbls = batch["lbls"].to("cuda")
                logits = model(feats, lbls[:, :-1], lang)
                val_loss += nn.functional.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE), lbls[:, 1:].reshape(-1), ignore_index=-100
                ).item()
                val_n += 1
        val_loss /= max(val_n, 1)
        print(f"  Val {lang}: loss={val_loss:.4f}")
        save_best(lang, val_loss, ep)
        del val_loader
        gc.collect(); torch.cuda.empty_cache()
        state["train_i"][lang] = 0
        state["val_i"][lang] = 0

    start_lang_i = 0
    state["epoch"] = ep + 1

print("\nTraining complete!")

In [ ]:
# CELL 6: Quick WER Check

@torch.no_grad()
def generate(model, audio_feat, lang, max_len=256):
    enc = model.encode(audio_feat)
    device = audio_feat.device
    dec_ids = torch.tensor([[tok.bos_token_id]], device=device)
    for _ in range(max_len):
        logits = model.adapters[lang](enc, dec_ids)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        dec_ids = torch.cat([dec_ids, next_token], dim=1)
        if next_token.item() == tok.eos_token_id:
            break
    return tok.decode(dec_ids[0].cpu().tolist(), skip_special_tokens=True)

@torch.no_grad()
def quick_wer(lang, n=20):
    model.eval()
    test_loader = make_loader(test_sets[lang], batch_size=1, shuffle=True)
    refs, hyps = [], []
    for i, batch in enumerate(test_loader):
        if i >= n: break
        feats = batch["feats"].to("cuda")
        ref_ids = batch["lbls"][0]
        ref_ids = ref_ids[ref_ids != -100]
        ref = tok.decode(ref_ids, skip_special_tokens=True)
        hyp = generate(model, feats, lang)
        refs.append(ref.lower().strip() if lang == "en" else ref.strip())
        hyps.append(hyp.lower().strip() if lang == "en" else hyp.strip())
    from jiwer import wer
    w = wer(refs, hyps)
    print(f"\n{lang.upper()} WER (quick, {n} samples): {w*100:.1f}%")
    for i in range(min(5, len(refs))):
        print(f"  REF: {refs[i][:80]}")
        print(f"  HYP: {hyps[i][:80]}")
    return w

for lang in LANGUAGES:
    quick_wer(lang, n=20)

In [ ]:
# CELL 7: Download Adapters

from google.colab import files
for lang in LANGUAGES:
    src = ADAPTER_DIR / f"cv_{lang}_best.pt"
    if src.exists():
        files.download(str(src))
        print(f"Downloaded: {src.name}")